# Basic Report Generation

This notebook will provide a basic overview of how to generate some simple reports using `pelagos-py`.

Start by doing some imports and initializing the pipeline.

In [ ]:
import xarray as xr
from pathlib import Path
import os

cwd = os.getcwd()
os.chdir(f"{cwd}/../../src")
from pelagos_py.pipeline import *

For this example, we'll use the Nelson example as found in other notebooks.

In [ ]:
input_dir = Path("../examples/data/OG1")
input_file = input_dir / "Nelson_646_R.nc"
if not input_dir.exists():
    input_dir.mkdir(parents=True)
if not input_file.exists():
    import requests
    response = requests.get("https://linkedsystems.uk/erddap/files/Public_OG1_Data_001/Nelson_20240528/Nelson_646_R.nc")
    if response.status_code == 200:
        with open(input_file, "wb") as f:
            f.write(response.content)
        print(f"Example file downloaded and written to {input_dir.resolve()}")
    else:
        print("File download failed")
dataset = xr.load_dataset("../examples/data/OG1/Nelson_646_R.nc")
dataset

Now build our pipeline of steps. In this example, we'll load `example_config_reporting.yaml` from the premade configs. This configuration file consists of 4 steps:
1. Load the data (`Load OG1`)
2. Apply quality control (`Apply QC`)
3. Export the data (`Data Export`)
4. Write the report (`Write Data Report`)

In [ ]:
#   Building the Pipe object of various steps
Pipe = Pipeline(config_path = r"../examples/configs/example_config_reporting.yaml")

And now let's check on each of the steps that we've defined in our pipeline `Pipe`.

In [ ]:
for i, step in enumerate(Pipe.steps):
    print(f"{i} {step["name"]}, \t {list(step["parameters"].keys())}")
    print(f"\t\t {list(step["parameters"].items())}")

The most important step for this notebook is the last one, which will assemble our logs and generate the report.

However, let's make one change. This Nelson dataset is closer to iceland, so let's tweak the final `extent` parameter, which defines an inset figure.

In [ ]:
Pipe.steps[3]["parameters"]["title"] = "Report Demo"
Pipe.steps[3]["parameters"]["fname"] = "demo.rst"
Pipe.steps[3]["parameters"]["extent"] = [5, -35, 45, 70]  #   This example is closer to Iceland. Wrap the inset figure around the data domain.

In [ ]:
for i, step in enumerate(Pipe.steps):
    print(f"{i} {step["name"]}, \t {list(step["parameters"].keys())}")
    print(f"\t\t {list(step["parameters"].items())}")

So now we've tweaked the pipeline to correctly run through. Now we can start the run.

*Note that in Jupyter notebooks, the figures can be spit out into the output. Not all of them are relevant, so the next cell's output can be kept collapsed.*

In [ ]:
Pipe.run()

The report generation step wrapped up at the very end. Since we didn't change anything related to the pipeline configuration itself, the `out_directory` should still take us to "../examples/data/OG1/testing/".

In [ ]:
print(Pipe.pipeline["out_directory"])
print(os.listdir(Pipe.pipeline["out_directory"]))

This has done a number of things. First, we can find all of the information about the run within our logfile - including how long it took to run particular steps. Using the default report template, this logfile is included.

Then it built a `conf.py` and a `demo.rst` file which Sphinx could use to assemble the final PDF.

Next, it has built a series of pictures, including QC results for all the columns with the "_QC" suffix by the end. These QC variables may/may not have been tested.

In [ ]:
from IPython.display import Image
Image(f"{Pipe.pipeline["out_directory"]}geographic.png")

As we can see in the log, the `Apply QC` step "Found QC columns for untested values" and there are a number of columns with QC columns that are not assessed.

In [ ]:
with open(Path(Pipe.pipeline['out_directory'] + Pipe.pipeline['log_file'])) as f:
    content = f.read()
    print(content)

Then Sphinx was run and everything was output to "../testing/_build/latex". This is where we can see the final formatted report, including copies of all the individual pieces that went into it.

In [ ]:
print(os.listdir(f"{Pipe.pipeline["out_directory"]}/_build/latex"))

Since we ran Sphinx's `build latexpdf`, it went into the default Sphinx _build/ location. We can easily find it at the link below.

In [ ]:
#   Link to final PDF
path = f"{Pipe.pipeline['out_directory']}_build/latex/Report_Demo.pdf"
print(os.path.abspath(path))